# ריצת הייצור: מיליון סימולציות וניתוח אירועים נדירים

מחברת זו מפעילה את ריצת הייצור המאושרת עבור StarLadder בשני seeds בלתי־תלויים. הגדלת המדגם למיליון מסלולים מצמצמת את רעש Monte Carlo גם עבור צירופי גמר ופודיומים נדירים, אך אינה מצמצמת אי־ודאות שמקורה במודל, ב־veto עתידי או במידע שלא נמצא בדאטה.

## בדיקת קדם־טיסה

לפני ריצת המיליון מופעלים 5,000 מסלולים פעמיים עם אותו seed: פעם דרך Booster ומטריצות NumPy, ופעם דרך `CalibratedClassifierCV` ו־DataFrame עם שמות הפיצ'רים. נדרשת זהות מלאה של מוני השלבים ושל כל מוני האנליטיקה. בנוסף, הנתיב המהיר מסרב להיבנות אם סדר הפיצ'רים השמור ב־Booster שונה מן הסדר הנעול.

## אנליטיקה מתקדמת

בכל מסלול נשמרים matchup הגמר בסדר אלפביתי, הסגנית, הפודיום המדויק והגעות לגמר של קבוצות שהסתברות האליפות הכוללת שלהן נמוכה מ־10%. המקום השלישי הוא המפסיד במשחק ה־Lower path האחרון—ה־Consolidation Final—שקובע מי מצטרף למנצחת ה־Upper Bracket בגמר הגדול. המונים נאספים רק לאחר שהמשחקים הוכרעו ולכן אינם משפיעים על הסתברויות המודל או על עדכוני ה־Elo.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.run_starladder_1m import main

production_runs = main()

Last historical map: 2026-06-21 18:00:00
{'mouz': 1686.324, 'nrg': 1351.652, 'vitality': 1868.139, 'magic': 1577.034, 'natus vincere': 1776.149, 'aurora': 1718.412, 'furia': 1739.099, 'mibr': 1528.695}


Pre-flight passed exactly: fast=4.914s, slow=8.378s


Double-Elimination Monte Carlo:   0%|          | 0/1000000 [00:00<?, ?iter/s]

seed=42 execution_seconds=1177.713
         Team  P(QF) P(SF) P(Final) P(Champion) SE(Champion)
     Vitality 100.0% 87.7%    67.9%       44.4%         0.0%
Natus Vincere 100.0% 73.1%    42.3%       20.5%         0.0%
        FURIA 100.0% 70.5%    34.5%       14.8%         0.0%
       Aurora 100.0% 55.9%    22.8%        9.4%         0.0%
         MOUZ 100.0% 58.8%    23.6%        8.9%         0.0%
        magic 100.0% 30.3%     5.5%        1.3%         0.0%
         MIBR 100.0% 17.8%     3.0%        0.6%         0.0%
          NRG 100.0%  5.9%     0.4%        0.0%         0.0%

Top 3 — Grand Final Matchups
                  Matchup Probability
Natus Vincere vs Vitality    24.9907%
        FURIA vs Vitality    19.3101%
       Aurora vs Vitality    12.6870%

Top 3 — Runner-Up
         Team Probability
     Vitality    23.4931%
Natus Vincere    21.8461%
        FURIA    19.6830%

Top 3 — Exact Podium [1st, 2nd, 3rd]
                          Podium Probability
Vitality > Natus Vincere > F

Double-Elimination Monte Carlo:   0%|          | 0/1000000 [00:00<?, ?iter/s]

seed=99 execution_seconds=1367.154
         Team  P(QF) P(SF) P(Final) P(Champion) SE(Champion)
     Vitality 100.0% 87.7%    67.9%       44.4%         0.0%
Natus Vincere 100.0% 73.1%    42.4%       20.6%         0.0%
        FURIA 100.0% 70.3%    34.5%       14.8%         0.0%
       Aurora 100.0% 56.0%    22.7%        9.4%         0.0%
         MOUZ 100.0% 58.8%    23.6%        8.9%         0.0%
        magic 100.0% 30.3%     5.5%        1.3%         0.0%
         MIBR 100.0% 17.8%     3.0%        0.6%         0.0%
          NRG 100.0%  5.9%     0.4%        0.0%         0.0%

Top 3 — Grand Final Matchups
                  Matchup Probability
Natus Vincere vs Vitality    25.0332%
        FURIA vs Vitality    19.2867%
       Aurora vs Vitality    12.7022%

Top 3 — Runner-Up
         Team Probability
     Vitality    23.5626%
Natus Vincere    21.8010%
        FURIA    19.6762%

Top 3 — Exact Podium [1st, 2nd, 3rd]
                          Podium Probability
Vitality > Natus Vincere > F

## שער היציבות והיצוא

ההשוואה בין seed 42 ל־seed 99 אינה נעצרת בהסתברות האליפות. נבדק גם ההפרש המרבי בהתפלגות matchups של Grand Final, סגניות, פודיומים מדויקים וריצות סינדרלה. רק אם כל הקטגוריות נמצאות בתוך רף של 0.5 נקודת אחוז נכתבים קובצי CSV, JSON ו־metadata אטומיים. ה־metadata כולל את כל התפלגויות האנליטיקה, זמני הריצה, חיתוך הזמן וטביעת SHA־256 של המודל.